## 1. Basic Tasks 

# Q1 Lakehouse Architecture Overview

- **Data Warehouse**: Optimized for structured data and fast SQL analytics, but costly and inflexible for raw, unstructured, or streaming data.
- **Data Lake**: Highly scalable and inexpensive storage for raw data (CSV, JSON, Parquet), but lacks ACID transactions, schema enforcement, and slow query performance.
- **Lakehouse (Databricks + Delta Lake)**: Combines the low-cost storage and flexibility of a data lake with the reliability, ACID transactions, schema enforcement, and high-performance querying of a traditional data warehouse.

## Q2 Create a Delta Table with Sample Product Data

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa.ass1.cyntexa_products (
    product_id INT,
    name STRING,
    category STRING,
    price DOUBLE
);

-- Insert 10 sample rows of product data
INSERT INTO cyntexa.ass1.cyntexa_products VALUES 
(1, 'Wireless Mouse', 'Electronics', 25.99),
(2, 'Mechanical Keyboard', 'Electronics', 79.99),
(3, 'USB-C Cable', 'Electronics', 12.50),
(4, 'Ergonomic Chair', 'Furniture', 199.99),
(5, 'Standing Desk', 'Furniture', 450.00),
(6, 'Coffee Mug', 'Home', 8.99),
(7, 'Stainless Steel Bottle', 'Home', 15.00),
(8, 'Notebook', 'Stationery', 4.50),
(9, 'Gel Pens (Pack of 10)', 'Stationery', 6.00),
(10, 'Noise Canceling Headphones', 'Electronics', 149.99);

## Q3 Run 3 Data Operations & View History

In [0]:
%sql
-- Statement 1: INSERT a new product 
INSERT INTO cyntexa.ass1.cyntexa_products VALUES
(11, 'Desk Lamp', 'Home', 29.99);

-- Statement 2: UPDATE an existing price 
UPDATE cyntexa.ass1.cyntexa_products 
SET price = 69.99
WHERE product_id = 2;

-- Statement 3: UPDATE category prices with a discount 
UPDATE cyntexa.ass1.cyntexa_products
SET price = price * 0.90
WHERE category = 'Stationery';

-- Inspect the transaction log history
DESCRIBE HISTORY cyntexa.ass1.cyntexa_products;

## Q4 Time Travel Query (Data Analyst Task)

In [0]:
%sql
-- 1. Query table state BEFORE updates (Version 1)
SELECT * FROM cyntexa.ass1.cyntexa_products VERSION AS OF 1 WHERE product_id IN (2,8,9,11);

In [0]:
%sql
-- 2. Query current table state (Latest Version)
SELECT * FROM cyntexa.ass1.cyntexa_products WHERE product_id IN (2,8,9,11);

## 2. Intermediate Tasks  

## Schema Enforcement vs Schema Evolution

In [0]:
%sql
-- Step 1: Demonstrate Schema Enforcement (This will intentionally FAIL)
-- Trying to insert a record with an unexpected extra column 'discount_percent'
INSERT INTO cyntexa.ass1.cyntexa_products VALUES
(12, 'Ergonomic Mousepad', 'Electronics', 15.99, 0.10);

In [0]:

# A metadata mismatch was detected when writing to the Delta table. SQLSTATE: 42KDG
# Step 2: Enable Schema Evolution via MergeSchema using PySpark
# Delta SQL INSERT does not allow inline mergeSchema, so we use PySpark
from pyspark.sql.types import StringType , StructField , IntegerType , StringType , DoubleType

# Define new data containing an extra column: "discount_percent"
new_data = [(12, 'Ergonomic Mousepad', 'Electronics', 15.99, 0.10)]

schema = StructType(
    [
        StructField("product_id" , IntegerType()),
        StructField("name" , StringType()),
        StructField("category" , StringType()),
        StructField("price" , DoubleType()),
        StructField("discount_percent" , DoubleType())
    ]
)

new_df = spark.createDataFrame(new_data , schema)

# Write using option("mergeSchema", "true") to update table structure on write
new_df.write.mode("append")\
.option("mergeSchema" , "true")\
.saveAsTable("cyntexa.ass1.cyntexa_products")

## Bad Update Simulation & Time Travel Recovery

In [0]:
%sql
-- Step 1: Simulate a "Bad Update" (Accidentally zero out all prices)
UPDATE cyntexa.ass1.cyntexa_products
SET price = 0.00;



In [0]:
%sql
-- Step 2: Record the safe state version number (Check your current version)
DESCRIBE HISTORY cyntexa.ass1.cyntexa_products;

In [0]:
%sql
-- Step 3: Inspect table to confirm corruption
SELECT * FROM cyntexa.ass1.cyntexa_products LIMIT 5;

In [0]:
%sql
-- Step 4: Time Travel - Inspect table using VERSION AS OF
SELECT * FROM cyntexa.ass1.cyntexa_products VERSION AS OF 6 LIMIT 5;
    

## USING "VERSION AS OF"

In [0]:
%sql
-- Step 5: Execute RESTORE command to rollback the bad update permanently
RESTORE TABLE cyntexa.ass1.cyntexa_products TO VERSION AS OF 6 

In [0]:
%sql
-- Step 6: Verify table is fully recovered
SELECT * FROM cyntexa.ass1.cyntexa_products LIMIT 5

## BAD UPDATE AGAIN TO IMPLEMENT "TIME STAMP"

In [0]:
%sql
UPDATE cyntexa.ass1.cyntexa_products
SET price = 0.00

In [0]:
%sql 
SELECT * FROM cyntexa.ass1.cyntexa_products LIMIT 5 

In [0]:
%sql
-- Step 2: Record the safe state Timestamp (Check your current version)
DESCRIBE HISTORY cyntexa.ass1.cyntexa_products;

In [0]:
%sql
RESTORE cyntexa.ass1.cyntexa_products TIMESTAMP AS OF '2026-08-21T09:34:55.000+00:00'

In [0]:
%sql
-- Data Verification Query
SELECT * FROM cyntexa.ass1.cyntexa_products

## Q7 Why ACID Transactions Matter for Retail Analytics

Imagine two retail store teams updating our product inventory table at the exact same second:
- **Pipeline A** updates inventory counts from yesterday's online sales.
- **Pipeline B** updates product prices from the promotion team.

Without **ACID Transactions**, these two pipelines would overwrite each other's files mid-write. Analysts running nightly revenue reports would see partial, corrupted, or "half-baked" numbers—like a product with an updated price but missing sales data.

### Delta Lake's ACID Properties :
1. **Atomicity (All-or-Nothing Writes)**: If a nightly job crashes 90% of the way through, none of its partial data is saved. Reports stay clean.
2. **Consistency & Schema Enforcement**: Ensures incorrect formats (e.g., entering text into a price column) are rejected before ruining the reporting table.
3. **Isolation (Concurrent Safety)**: Pipeline A and Pipeline B can run simultaneously without seeing or corrupting each other's half-finished updates.
4. **Durability**: Once a transaction completes, it is permanently saved and immediately ready for nightly analytics dashboards.

## 3. Advanced Tasks 

# Q8 Design Note: Replacing Legacy Nightly Warehouse Load with a Lakehouse Architecture

## Current State & Bottlenecks
Currently, Cyntexa relies on a legacy nightly ETL batch load into a relational data warehouse. This approach faces key risks:
1. **Long Loading Windows & Failures**: If a batch pipeline fails mid-execution, partial data writes leave reports corrupted, requiring full manual cleanups or lengthy full restores.
2. **Concurrent Write Lockouts**: Analyst reporting jobs running during the batch window lock tables, causing jobs to fail or stall.
3. **High Storage & Compute Costs**: Scaling compute to process raw batch files alongside analytical reporting requires expensive data warehouse scaling.

## Lakehouse Migration Strategy (Databricks + Delta Lake)
By adopting Delta Lake, Cyntexa moves from rigid batch loading to continuous/micro-batch processing using ACID transactions and Time Travel.

### How Delta Lake Reduces Operational Risk:
- **ACID Transactions (Atomicity & Isolation)**:
  - Delta Lake records all writes as an atomic commit in the `_delta_log`. 
  - If a nightly job crashes mid-write, partial files are ignored during queries. The table remains in its last known clean state without manual intervention.
  - Concurrent writes use **Optimistic Concurrency Control (OCC)**, allowing analysts to run live reports while production pipelines write data without table-level locks.
- **Time Travel & Auditing**:
  - Every update creates a immutable data version. If bad source data enters the pipeline, data engineers can instantly execute `RESTORE TABLE cyntexa_products TO VERSION AS OF <safe_version>` rather than running costly pipeline backfills.

## Q9 How the Delta Transaction Log Resolves Conflict:
**Optimistic Concurrency Control (OCC)**: Delta Lake assumes conflicts are rare. Both writers read the current latest version (e.g., Version 0) and attempt to write new data files to storage.

**First-Committer Wins**: Whichever pipeline writes its commit JSON file (e.g., 00000000000000000001.json) to the `_delta_log` directory first successfully establishes Version 1.

**Automatic Retry on Conflict**: The second writer detects that Version 1 was created while it was writing. Delta Lake automatically checks if the second write conflicts with the first write:

Since both writes are simple INSERT operations (append-only), there is no row-level overlap. Delta Lake automatically retries and commits the second write as Version 2.

If both writers attempted to UPDATE or DELETE the exact same row simultaneously, Delta Lake would throw a `ConcurrentAppendException` or `ConcurrentTransactionException`, rolling back the conflicting write safely to preserve data integrity.

# Q10 Comparison Memo: Databricks Lakehouse vs. Traditional Data Warehouse

---

### Key Advantages for Analysts

1. **Instant Time Travel & Risk-Free Data Exploration**
   - **Advantage**: Analysts can query historical snapshots using `VERSION AS OF` or `TIMESTAMP AS OF` without relying on database backups.
   - **Impact**: Analysts can easily perform trend audits, track historical price changes, and verify row-level changes over time. Mistakes can be reverted instantly using `RESTORE`.

2. **Access to Raw, Semi-Structured, and Structured Data in One Place**
   - **Advantage**: Traditional warehouses require raw JSON/CSV data to be heavily transformed via ETL before querying. A Lakehouse allows querying raw logs, JSON blobs, and structured relational tables directly using native SQL.
   - **Impact**: Dramatically reduces time-to-insight for new reporting metrics without waiting on upstream data engineering build cycles.

3. **Zero-Copy Data Sharing & Concurrent Performance**
   - **Advantage**: Readers (analysts executing queries) never block writers (pipelines updating tables) and vice versa. 
   - **Impact**: Heavy ETL processing during overnight windows won't degrade dashboard load times or query speed for reporting analysts.

---

### Key Tradeoff to Watch For

* **Storage Hygiene & Table Maintenance Requirements (VACUUM / OPTIMIZE)**
  - **Tradeoff**: Because Delta Lake maintains full version history by keeping older Parquet files, storage size can grow rapidly if stale data files are not periodically cleaned.
  - **Mitigation**: Data teams must schedule periodic maintenance queries (`OPTIMIZE` to combine small files and `VACUUM` to delete old data files beyond the retention period) to ensure query performance and control storage costs.